In [1]:
import pandas as pd
import numpy as np

In [4]:
# Load annotation file
annot_path = "/Users/esther/Documents/Emory/Varvel/honors_thesis/proteomic_clock/data/EMBOJ-2025-120525_SD_Fig1B.xlsx"
annot = pd.read_excel(annot_path, sheet_name="Brain_Proteins", header=1)

# Load trajectory data
combined = pd.read_excel("/Users/esther/PycharmProjects/ad_proteomic_clock/results/3M_DEP_trajectory.xlsx", sheet_name="trajectory_ranked", index_col=0)

# Check
print(annot.columns.tolist())
print(combined.shape)
print(combined.columns.tolist())
print(combined['pattern'].value_counts())

annot = annot.set_index("UniProt Accession")

['UniProt Accession', 'UniProt Protein Name', 'Gene Symbol', '8M', '8M.1', '8M.2', '8M.3', '8M.4', '8M.5', '8M.6', '8M.7', '14M', '14M.1', '14M.2', '14M.3', '14M.4', '14M.5', '14M.6', '14M.7', '14M.8', '20M', '26M', '20M.1', '26M.1', '20M.2', '26M.2', '20M.3', '26M.3', '20M.4', '26M.4', '20M.5', '20M.6', '3M', '5M', '3M.1', '5M.1', '3M.2', '5M.2', '3M.3', '5M.3', '5M.4', '3M.4', '5M.5', '5M.6', '3M.5', '5M.7', '5M.8', '3M.6']
(825, 12)
['n_significant', 'pattern', 'logFC_5M', 'pval_5M', 'logFC_8M', 'pval_8M', 'logFC_14M', 'pval_14M', 'logFC_20M', 'pval_20M', 'logFC_26M', 'pval_26M']
pattern
transient_middle    451
late_onset          169
early_then_drop     109
intermittent         91
always                5
Name: count, dtype: int64


In [5]:
# Filter to late_onset proteins only
late_onset = combined[combined['pattern'] == 'late_onset'].copy()
print(f"{len(late_onset)} late_onset proteins") # 169

169 late_onset proteins


In [10]:
# Check/filter FC direction consistency for significant timepoints
timepoints = ['5M', '8M', '14M', '20M', '26M']
logfc_cols = [f'logFC_{t}' for t in timepoints]

def check_direction_cons(row):
    sig_mask = row[[f'pval_{t}' for t in timepoints]].values < 0.05
    vals = pd.Series(row[logfc_cols].values[sig_mask]).dropna()
    if len(vals) == 0:
        return np.nan
    sign = np.sign(vals)
    return (sign == sign.iloc[0]).all()

late_onset['consistent_direction'] = late_onset.apply(check_direction_cons, axis=1)
print(late_onset['consistent_direction'].value_counts())

consistent_direction
True    169
Name: count, dtype: int64


In [11]:
# Map to protein name and gene symbol
late_onset = late_onset.join(annot[['UniProt Protein Name', 'Gene Symbol']], how='left')

ValueError: columns overlap but no suffix specified: Index(['UniProt Protein Name', 'Gene Symbol'], dtype='str')

In [12]:
# Sort: directional consistency, n_sig
late_onset = late_onset.sort_values(['consistent_direction', 'n_significant'], ascending=[False, False])

col_order = ['UniProt Protein Name', 'Gene Symbol','consistent_direction', 'n_significant', 'pattern'] + logfc_cols + [f'pval_{t}' for t in timepoints]
late_onset = late_onset[col_order]

print(late_onset.head(5))

                                    UniProt Protein Name Gene Symbol  \
protein                                                                
P62071                        Ras-related protein R-Ras2       Rras2   
P70677                                         Caspase-3       Casp3   
P97492               Regulator of G-protein signaling 14       Rgs14   
Q64727                                          Vinculin         Vcl   
Q8R105   Vacuolar protein sorting-associated protein 37C      Vps37c   

         consistent_direction  n_significant     pattern  logFC_5M  logFC_8M  \
protein                                                                        
P62071                   True              4  late_onset -0.145789 -0.538167   
P70677                   True              4  late_onset -0.632170 -0.610055   
P97492                   True              4  late_onset -0.125651 -0.258340   
Q64727                   True              4  late_onset  0.159102  0.277486   
Q8R105         

In [13]:
# Save to Excel
with pd.ExcelWriter("/Users/esther/PycharmProjects/ad_proteomic_clock/results/3M_DEP_trajectory.xlsx", mode='a', if_sheet_exists='replace') as writer:
    late_onset.to_excel(writer, sheet_name="late_onset_annotated")